In [1]:
# For data access and handling
import ee # earthengine-api
import xarray
import rioxarray

# For plotting
import matplotlib.pyplot as plt
import cartopy.feature as cfeature
import cartopy.crs as ccrs
import cmocean # Colormap
import pandas as pd
import time

import hvplot.xarray

import gc

In [ ]:
# 強制重新進行身份驗證流程（更換帳號一定要做）
# ee.Authenticate(force=True)
# 用以下的code才可以使用ee.batch.Export.image.toDrive()，存到Google Drive
ee.Authenticate(quiet=True)
ee.Initialize(project="bigdata-493602")

#### 資料抓取

In [3]:
### 前面的code會遇到「影像全黑」或是「部分黑色」的情況，這段在解決這個問題
### 原因可能是.first()只抓一張圖幅，而該圖幅並不一定涵蓋ROI
csv_path = "/home/chunen/nas/bigdata/final/kurosiwo/kurosiwo_bboxes_times_aoi.csv"

df = pd.read_csv(csv_path)
df_remain = df.iloc[:66]
# selected_filename = [174]
# df_select = df[df['filename'].isin(selected_filename)]

fail_found = []

tasks = []

for _, row in df_remain.iterrows():

    filename = row["filename_x"]
    west, south, east, north = row["west"], row["south"], row["east"], row["north"]
    target_time = pd.to_datetime(row["time1"])

    print(f"Processing {filename}...")

    roi = ee.Geometry.Rectangle([west, south, east, north])

    # 🔥 縮小時間窗（很重要）
    start = target_time - pd.Timedelta(days=3)
    end   = target_time + pd.Timedelta(days=3)

    ic = (
        ee.ImageCollection("COPERNICUS/S2_SR_HARMONIZED")
        .filterDate(start.strftime("%Y-%m-%d"), end.strftime("%Y-%m-%d"))
        .filterBounds(roi)
    )
    # ❗ 避免空集合
    if ic.size().getInfo() == 0:
        print(f"⚠️ No image for {filename}")
        time_str = target_time.strftime("%Y%m%dT%H%M%S")
        fail_found.append(f"{filename}_{time_str}")
        continue
    # 🔥 計算時間差
    def add_time_diff(img):
        diff = ee.Number(img.date().millis()).subtract(target_time.timestamp() * 1000).abs()
        return img.set("time_diff", diff)
    # 1. 取得這包影像中最接近的影像（用來得知基準時間與命名）
    closest_img = ic.map(add_time_diff).sort("time_diff").first()
    actual_time = ee.Date(closest_img.get("system:time_start")).format("YYYYMMdd'T'HHmmss")
    time_str = actual_time.getInfo()
    out_name = f"{filename}_{time_str}"
    # ⭐⭐ 終極修正：將整 6 天的影像照時間差「由大到小」(False) 排序並全部拼接
    # GEE 的 mosaic() 原理是：排序越後面的影像會疊在越「上層」。
    # 這樣一來：時間最接近 (time_diff 最小) 的會排在最後被加上去，因此能覆蓋在最上面。
    # 它的邊界或黑塊 (NoData) 則會由時間稍微遠一點（底下）的影像完美補滿！
    ic_sorted_for_mosaic = ic.map(add_time_diff).sort("time_diff", False).select(['B4', 'B3', 'B2', 'B8'])
    img_mosaic = ic_sorted_for_mosaic.mosaic()
    # 🚀 GEE Export（不變，請繼續使用）
    # 可至 https://code.earthengine.google.com/tasks 查看任務狀態
    task = ee.batch.Export.image.toDrive(
        image=img_mosaic,
        description=out_name,
        folder="GEE_S2_COG",
        fileNamePrefix=out_name,
        region=roi,
        scale=10,                  
        crs="EPSG:3857",           
        fileFormat="GeoTIFF",
        formatOptions={
            "cloudOptimized": True  
        },
        maxPixels=1e13             
    )

    task.start()
    tasks.append(task)

    print(f"Started task: {out_name}")

    # ⭐ 避免 GEE API rate limit
    time.sleep(0.5)

print(f"Total tasks: {len(tasks)}")
print(f'fail_found: {fail_found}')

Processing 1111002...
Started task: 1111002_20200622T093724
Processing 1111002...
Started task: 1111002_20200707T093720
Processing 1111002...
Started task: 1111002_20200915T093720
Processing 1111003...
Started task: 1111003_20191107T074645
Processing 1111003...
Started task: 1111003_20191117T074639
Processing 1111003...
⚠️ No image for 1111003
Processing 1111004...
Started task: 1111004_20170723T170538
Processing 1111004...
Started task: 1111004_20170805T171528
Processing 1111004...
Started task: 1111004_20170828T172538
Processing 1111005...
Started task: 1111005_20191117T070434
Processing 1111005...
Started task: 1111005_20191127T070432
Processing 1111005...
Started task: 1111005_20200126T070426
Processing 1111006...
Started task: 1111006_20191130T071420
Processing 1111006...
Started task: 1111006_20200119T071400
Processing 1111006...
Started task: 1111006_20200129T071414
Processing 1111007...
Started task: 1111007_20190720T051136
Processing 1111007...
Started task: 1111007_20190720T0

#### 檢查影像

In [ ]:
import geemap
# --- 開始使用 geemap 視覺化 ---
Map = geemap.Map()
# 視覺化參數：Sentinel-2 Level-2A reflectance 數值大約在 0~10000 之間。
# 設定 max 為 3000 可以有很好的對比度 (如果設太高畫面還是會很黑)
viz_params = {
    'bands': ['B4', 'B3', 'B2'], # 真彩色 RGB
    'min': 0,
    'max': 3000, 
    'gamma': 1.2
}
# 把地圖中心移動到這塊 roi，縮放等級大概 11 左右
Map.centerObject(roi, 11)
# 加入我們疊合好的影像
Map.addLayer(img_mosaic.clip(roi), viz_params, f"Mosaic for {filename}")
# 可選：把 ROI 框出來，看看 ROI 到底長在哪裡
Map.addLayer(ee.FeatureCollection([ee.Feature(roi)]), {'color': 'red'}, 'Bounds ROI')
# 顯示地圖S
Map

Map(center=[55.297842567282146, 21.44400901799945], controls=(WidgetControl(options=['position', 'transparent_…